# Explore the local DuckDB warehouse

This notebook queries `data/warehouse.duckdb`, the file the Docker container builds from the
Snowflake-managed Iceberg tables on S3 (`products`, `customers`, `orders`) and that Power BI reads.

* The connection is **read-only**, so the container can keep refreshing in the background.
* Everything lives in schema `reporting`: raw tables `products`, `customers`, `orders`,
  views `v_sales`, `dim_date`, `v_daily_revenue`, and `refresh_log`.
* Run `./duck.sh notebook` from the repo root to start JupyterLab with the right packages.

Column names in the raw tables are UPPER CASE (that is how Snowflake wrote the Iceberg schema).
DuckDB identifiers are case-insensitive, so `order_id` and `ORDER_ID` both work.

In [ ]:
import os
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

DB = os.environ.get("DUCKDB_PATH", "../data/warehouse.duckdb")
con = duckdb.connect(DB, read_only=True)

def q(sql: str) -> pd.DataFrame:
    """Run SQL and return a pandas DataFrame."""
    return con.sql(sql).df()

print(f"DuckDB {duckdb.__version__}  ->  {os.path.abspath(DB)}")

## What is in the warehouse?

In [ ]:
q("""
SELECT t.table_name,
       t.table_type,
       CASE WHEN t.table_type = 'BASE TABLE' THEN (
            SELECT estimated_size FROM duckdb_tables() d
            WHERE d.schema_name = t.table_schema AND d.table_name = t.table_name)
       END AS est_rows
FROM information_schema.tables t
WHERE t.table_schema = 'reporting'
ORDER BY t.table_type, t.table_name
""")

In [ ]:
# When was each table last pulled from S3, and from which Iceberg snapshot?
q("""
SELECT refreshed_at, table_name, row_count, snapshot_id, snapshot_ts, seconds, source
FROM reporting.refresh_log
ORDER BY refreshed_at DESC, table_name
LIMIT 12
""")

## Ad-hoc SQL

`con.sql(...)` prints a nicely formatted relation; `q(...)` gives you a DataFrame. Edit freely.

In [ ]:
con.sql("""
SELECT order_id, order_date, customer_name, region, product_name, quantity, unit_price, revenue
FROM reporting.v_sales
ORDER BY order_date DESC
LIMIT 10
""")

In [ ]:
q("""
SELECT count(*)                    AS orders,
       count(DISTINCT customer_id) AS customers,
       sum(quantity)               AS units,
       round(sum(revenue), 2)      AS revenue,
       round(avg(revenue), 2)      AS avg_order_value,
       min(order_day)              AS first_order,
       max(order_day)              AS last_order
FROM reporting.v_sales
""")

## Charts

In [ ]:
# Small styling helper: thin marks, recessive axes/grid, one hue per chart.
BLUE = "#2a78d6"
INK, MUTED, GRID = "#1f1f1e", "#6b6a66", "#e6e5e0"

def style(ax, title, ylabel=None, value_axis="y"):
    ax.set_title(title, loc="left", fontsize=12, fontweight=600, color=INK, pad=12)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.grid(axis=value_axis, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(length=0, colors=MUTED)
    fmt = lambda v, _: f"{v/1000:,.0f}k" if abs(v) >= 1000 else f"{v:,.0f}"
    (ax.yaxis if value_axis == "y" else ax.xaxis).set_major_formatter(fmt)
    if ylabel:
        ax.set_ylabel(ylabel, color=MUTED)

plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "text.color": INK, "axes.labelcolor": MUTED})

In [ ]:
df = q("""
SELECT region, round(sum(revenue), 2) AS revenue, count(*) AS orders
FROM reporting.v_sales
GROUP BY region
ORDER BY revenue DESC
""")

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.bar(df["region"], df["revenue"], color=BLUE, width=0.6)
style(ax, "Revenue by region")
plt.show()
df

In [ ]:
df = q("""
SELECT date_trunc('month', order_day) AS month, round(sum(revenue), 2) AS revenue
FROM reporting.v_sales
GROUP BY 1
ORDER BY 1
""")

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(df["month"], df["revenue"], color=BLUE, linewidth=2, marker="o", markersize=4)
style(ax, "Monthly revenue")
fig.autofmt_xdate()
plt.show()

In [ ]:
df = q("""
SELECT product_name, sum(quantity) AS units, round(sum(revenue), 2) AS revenue
FROM reporting.v_sales
GROUP BY product_name
ORDER BY revenue
""")

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.barh(df["product_name"], df["revenue"], color=BLUE, height=0.6)
style(ax, "Revenue by product", value_axis="x")
ax.tick_params(axis="y", labelcolor=INK)
plt.show()

## Top customers

In [ ]:
q("""
SELECT customer_name, region,
       count(*)               AS orders,
       sum(quantity)          AS units,
       round(sum(revenue), 2) AS revenue
FROM reporting.v_sales
GROUP BY customer_name, region
ORDER BY revenue DESC
LIMIT 10
""")

## Hand off to pandas

Anything you pull with `q()` is a normal DataFrame, so the usual pandas / scikit-learn / Excel
workflows apply.

In [ ]:
sales = q("SELECT * FROM reporting.v_sales")
sales.describe(include="all").T.head(12)

In [ ]:
# Export for colleagues (data/exports is created on demand)
os.makedirs("../data/exports", exist_ok=True)
out = "../data/exports/sales_by_region_month.csv"
q("""
SELECT region, date_trunc('month', order_day) AS month, round(sum(revenue), 2) AS revenue
FROM reporting.v_sales GROUP BY ALL ORDER BY 1, 2
""").to_csv(out, index=False)
print("wrote", out)

## Optional: query the Iceberg tables on S3 directly

The cell below re-uses the container's connection logic (`docker/refresh.py`) with the settings in
`docker/.env`, so you can compare the live S3 snapshot with the local copy. It needs AWS
credentials on this machine (the same profile or keys you put in `docker/.env`).

In [ ]:
import sys
from pathlib import Path

env_file = Path("../docker/.env")
if not env_file.exists():
    print("docker/.env not found - skip, or run ./duck.sh init and fill it in.")
else:
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip())
    if not os.environ.get("ICEBERG_S3_ROOT"):
        print("ICEBERG_S3_ROOT is empty in docker/.env - nothing to query.")
    else:
        sys.path.insert(0, "../docker")
        import refresh
        cfg = refresh.Config()
        live = duckdb.connect()
        refresh.setup_connection(live, cfg)
        for t in cfg.tables:
            n = live.sql(f"SELECT count(*) FROM {refresh.scan_sql(cfg, cfg.table_location(t))}").fetchone()[0]
            print(f"{t:<12} {n:>8} rows on S3")
        live.close()

In [ ]:
con.close()